# 01 · Define & Explore — MDM2 cleft prep, p53 sub-pockets, peptide metrics

**Standard slot:** *define & explore.* **For Project 09 this means:** clean the MDM2 N-terminal
domain, define the **p53-binding cleft** (the Phe19/Trp23/Leu26 sub-pockets) as the design site, write
down the binder metrics + cutoffs, and run a deterministic **mock** mini-run (a few linear + cyclic
peptides) as your "hello-world" (D0).

Run `00_setup.ipynb` first in this session. Peptides are small, so AF2/Boltz-2 scoring and Boltz-2
affinity on small inputs run on a **free T4**; a full macrocycle campaign prefers **Colab Pro** (see
`MANUAL.md §2`). Everything here runs on a no-GPU **mock** backend so you can build the plumbing
anywhere, then switch to the real backend on Colab.

## The peptide / macrocycle metrics, precisely

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| pLDDT | 0–100 | per-residue *local* confidence of the peptide | thermostability / protease stability / permeability |
| **pae_interaction** | Å | AF2/Boltz-2 error across the **peptide–MDM2 interface** (the key metric) | measured affinity |
| scRMSD | Å | designed-vs-predicted Cα-RMSD (self-consistency; noisier for short peptides) | binding/function |
| **boltz_affinity_score** | rel. | Boltz-2 **relative** affinity ranking signal | **a K_D** — never |
| shape complementarity | 0–1 | interface packing quality in the cleft | epitope correctness |
| cleft overlap | 0–1 | fraction of the p53 sub-pockets the peptide covers | a guarantee it displaces p53 |

The shared `"binder"` cutoffs: **scRMSD ≤ 2.5, pLDDT ≥ 80, pae_interaction ≤ 10, rosetta_dG ≤ −30,
sc ≥ 0.6.** `pae_interaction` is the single most important interface metric — but a low value is
*confidence*, **not** affinity. The **Boltz-2 affinity score is a relative RANK, not a K_D.** A passing
design is a **hypothesis** until synthesis + binding/stability assays — and **predicted affinity for
short peptides is unreliable: rank, don't trust.**

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Target prep + the p53 cleft

The design target is the **MDM2 N-terminal domain**, and the "hotspots" are the MDM2 residues lining
the **p53-binding cleft** — the three sub-pockets that bury p53 **Phe19 / Trp23 / Leu26**. Steering a
peptide there is what makes it a *p53-mimetic competitor*. Fetch the candidate complex with
`data/download_data.py` (**1YCR — verify on RCSB**), isolate the MDM2 chain, remove the p53
peptide/waters/heteroatoms, and read the cleft residues off the complex.

Below we just *declare* an EXAMPLE cleft set so the notebook runs end-to-end; **replace it with the
residues you derive from the actual MDM2–p53 interface** (numbering depends on the PDB you verify).

In [ ]:
import peptide_tools as pt

TARGET = "MDM2"                     # cleaned MDM2 N-terminal domain (you produce this from 1YCR)
# EXAMPLE cleft residues lining the p53 sub-pockets — VERIFY/REPLACE from the MDM2-p53 interface (data/README.md).
# These are placeholders so the plumbing runs; real numbering depends on the PDB chain you clean.
CLEFT = pt.parse_cleft("A54,A67,A73,A93,A100")   # EXAMPLE_DATA placeholder residues
print("target:", TARGET)
print("cleft :", CLEFT, " (EXAMPLE — replace with your verified p53-cleft residues from 1YCR)")

## 2 · Mock hello-world: a tiny linear + macrocyclic mini-run

`scripts/peptide_tools.py` exposes the design API: `design_peptide(..., cyclic=False|True)` for linear
and macrocyclic peptides, plus `boltz_affinity(...)` (the relative-ranking scorer). The **mock**
backend is deterministic and GPU-free so you can develop the plumbing. **Never report mock numbers as
real** — they are `SYNTHETIC` by construction, and the affinity score is a **relative rank, not a K_D**.

In [ ]:
# A few linear and a few macrocyclic designs, scored by mock AF2/Boltz-2. All numbers are SYNTHETIC.
lin = pt.design_peptide(TARGET, CLEFT, length=12, cyclic=False, n=3, tool="mock")
cyc = pt.design_peptide(TARGET, CLEFT, length=12, cyclic=True,  n=3, tool="mock")
pt.score_designs(lin, tool="mock")
pt.score_designs(cyc, tool="mock")

d = cyc[0]
print("example MACROCYCLE design:")
print("  id    :", d.design_id)
print("  len   :", d.length, "aa   cyclic:", d.cyclic)
print("  seq   :", d.sequence)
print("  pae_interaction =", d.pae_interaction, " scrmsd =", d.scrmsd, " sc =", d.shape_complementarity)
print("  boltz_affinity_score =", d.boltz_affinity_score, " (RELATIVE RANK, NOT a K_D; SYNTHETIC)")
print("  synthetic flag  :", d.synthetic, "->", d.notes[0])
print("\nReminder: switch tool='mock' -> 'boltzgen'/'evobind2'/'boltz' on Colab. See MANUAL.md §2.")

## 3 · Cleft-engagement proxy (does it cover the p53 sub-pockets?)

A peptide only *competes with p53* if it covers the p53 sub-pockets (Phe19/Trp23/Leu26). `cleft_overlap()`
is a geometry proxy (fraction of cleft residues contacted) — a teaching stand-in for the
p53-displacement assay in notebook 04. Higher ⇒ more likely to displace p53 (not a guarantee of
functional reactivation).

In [ ]:
for b in cyc[:3]:
    ov = pt.cleft_overlap(b.contact_residues, CLEFT)
    print(f"{b.design_id}: contacts {b.contact_residues} -> p53-cleft overlap = {ov} (SYNTHETIC)")

## Visualize a peptide–MDM2 cleft complex (py3Dmol)

Use this to eyeball a predicted peptide–MDM2 complex once you have a real PDB (from AF2/Boltz-2).

In [ ]:
import py3Dmol

def show_complex(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view.show()

# Example (after a real AF2/Boltz-2 prediction writes a complex PDB):
# show_complex("results/boltz/top_complex.pdb")
print("show_complex(pdb_path) ready.")

## D0 checklist
- [ ] MDM2–p53 accession verified on RCSB (**1YCR** is a candidate); MDM2 chain + N-terminal domain identified.
- [ ] Cleaned MDM2 target + **p53-cleft residue list** (the Phe19/Trp23/Leu26 sub-pockets; derived from the interface, not invented).
- [ ] One-paragraph definition of each metric **with** its "does not mean" note (esp. Boltz-2 score ≠ K_D).
- [ ] Reproduced mock mini-run (linear + cyclic) with metrics printed and flagged SYNTHETIC.
- [ ] Problem statement with measurable success criteria + controls; `LOG.md` entry (GPU, seed).

**Next:** `02_generate.ipynb` — the linear + macrocyclic peptide campaign.